# 03 — Q4 Spending Forecast

Three-model comparison for predicting October–December spending:
1. **Holt-Winters** Exponential Smoothing (baseline)
2. **XGBoost** with lag features (main model)
3. **LSTM** deep learning (exploration)

Evaluation: MAE ($), RMSE ($), R². Comparison table at the end.

In [16]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_pipeline
from src import features as feat
from src.models import spend_predictor as sp

pd.set_option('display.max_columns', 50)

In [17]:
# Load data
con, counts = load_pipeline(verbose=True)

Loading CSV files into DuckDB…
  ✓ account_dim: 18,070 rows
  ✓ statement_fact: 658,228 rows
  ✓ transaction_fact: 493,336 rows
  ✓ wrld_stor_tran_fact: 1,053,854 rows
  ✓ syf_id: 18,070 rows
  ✓ rams_batch_cur: 96,799 rows
  ✓ fraud_claim_case: 77 rows
  ✓ fraud_claim_tran: 202 rows
  ✓ transaction_base (union): 1,547,190 rows

Verifying row counts…
  ✓ account_dim: 18,070 rows (OK)
  ✓ statement_fact: 658,228 rows (OK)
  ✓ transaction_fact: 493,336 rows (OK)
  ✓ wrld_stor_tran_fact: 1,053,854 rows (OK)
  ✓ transaction_base: 1,547,190 rows (OK)
  ✓ syf_id: 18,070 rows (OK)
  ✓ rams_batch_cur: 96,799 rows (OK)
  ✓ fraud_claim_case: 77 rows (OK)
  ✓ fraud_claim_tran: 202 rows (OK)


## Data Preparation

In [18]:
# Build feature tables
customer_base = feat.build_customer_base(con)
customer_monthly = feat.build_customer_monthly(con)
customer_monthly = feat.add_lag_features(customer_monthly)
customer_monthly = feat.add_spend_to_limit_ratio(customer_monthly, customer_base)

print(f"customer_base: {customer_base.shape}")
print(f"customer_monthly: {customer_monthly.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

customer_base: (18070, 55)
customer_monthly: (663105, 22)


In [19]:
# Build Q4 feature matrix with temporal split
train_df, test_df, feature_cols, target_col = feat.build_q4_feature_matrix(
    customer_monthly, customer_base, cutoff_month='2024-10-01'
)

print(f"\nTemporal Split (CRITICAL — no data leakage):")
print(f"  Train: {train_df.shape[0]:,} rows (Jan-Sep)")
print(f"  Test:  {test_df.shape[0]:,} rows (Oct-Dec)")
print(f"  Features: {len(feature_cols)}")


Temporal Split (CRITICAL — no data leakage):
  Train: 47,880 rows (Jan-Sep)
  Test:  49,967 rows (Oct-Dec)
  Features: 28


## Model 1: Holt-Winters Exponential Smoothing (Baseline)

Per-account time series model using >=3 months of non-zero history.
Additive trend, no seasonal component (insufficient data for seasonal estimation).
Uses zero account attributes — pure time series baseline.

In [20]:
# Prepare per-account monthly series (training period only)
train_monthly = customer_monthly[customer_monthly['month'] < '2024-10-01'].copy()

print("Training Holt-Winters per account...")
hw_results = sp.train_holt_winters(train_monthly, min_months=3, forecast_periods=3)
print(f"  Forecasted {len(hw_results)} accounts")

Training Holt-Winters per account...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


  Forecasted 6173 accounts


In [21]:
# Evaluate against actual Q4 spend
actual_q4 = customer_monthly[customer_monthly['month'] >= '2024-10-01'].groupby(
    'current_account_nbr'
)['total_spend'].sum().reset_index()
actual_q4.columns = ['current_account_nbr', 'actual_q4_spend']

hw_eval = hw_results.merge(actual_q4, on='current_account_nbr', how='inner')
hw_metrics = sp.evaluate_predictions(
    hw_eval['actual_q4_spend'].values,
    hw_eval['hw_predicted_q4_spend'].values,
    "Holt-Winters"
)

  Holt-Winters: MAE=$3176.43  RMSE=$6291.57  R²=0.6125


## Model 2: XGBoost with Lag Features (Main Model)

Temporal train/test split: Jan-Sep → Oct-Dec (NO random split — prevents leakage).
Uses lag features, rolling stats, static account attributes, and sales trend.

In [22]:
print("\nTraining XGBoost spend predictor...")
xgb_model, xgb_preds, xgb_metrics = sp.train_xgboost_spend(
    train_df, test_df, feature_cols, target_col,
    save_path='outputs/saved_models/xgb_spend_predictor.joblib'
)


Training XGBoost spend predictor...
  XGBoost: MAE=$32.15  RMSE=$172.22  R²=0.9906
  → Model saved to outputs/saved_models/xgb_spend_predictor.joblib


In [24]:
# Feature importance
import xgboost
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True)

fig = px.bar(importance.tail(15), x='importance', y='feature', orientation='h',
             title='XGBoost Q4 Predictor — Top 15 Feature Importances',
             color='importance', color_continuous_scale='Viridis')
fig.update_layout(height=500, width=700, template='plotly_white', yaxis_title='')
fig.write_image("outputs/figures/xgb_feature_importance.png", scale=2)
fig.show()

## Model 3: LSTM (Deep Learning Exploration)

2-layer LSTM with dropout. Sequences: 9 months → predict Q4 total.
Only accounts with ≥12 months of history.
Simple feature set (total_spend, prev_balance) — complex features hurt LSTM on small datasets.

In [25]:
print("\nBuilding LSTM sequences...")
X_lstm, y_lstm, acct_ids = sp.build_lstm_sequences(
    customer_monthly, seq_length=9, min_history=12
)

print(f"  Sequences: {X_lstm.shape}")
print(f"  Targets: {y_lstm.shape}")


Building LSTM sequences...
  Sequences: (12135, 9, 2)
  Targets: (12135,)


In [26]:
print("\nTraining LSTM...")
if len(X_lstm) > 0:
    lstm_model, lstm_preds, lstm_metrics, y_lstm_test = sp.train_lstm_spend(
        X_lstm, y_lstm, test_size=0.2,
        save_path='outputs/saved_models/lstm_spend_predictor.keras'
    )
else:
    print("  Not enough sequences for LSTM training")
    lstm_metrics = {'model': 'LSTM', 'MAE ($)': float('nan'), 
                    'RMSE ($)': float('nan'), 'R²': float('nan')}


Training LSTM...
Epoch 61: early stopping
Restoring model weights from the end of the best epoch: 46.
  LSTM: MAE=$298.93  RMSE=$1058.49  R²=0.7627
  → LSTM model saved to outputs/saved_models/lstm_spend_predictor.keras


## Model Comparison

In [27]:
all_metrics = [hw_metrics, xgb_metrics]
if 'lstm_metrics' in dir() and lstm_metrics:
    all_metrics.append(lstm_metrics)

comparison = sp.comparison_table(all_metrics)
print("\n" + "="*60)
print("  Q4 SPENDING FORECAST — MODEL COMPARISON")
print("="*60)
print(comparison.to_string())
print()

# Identify best model
best = comparison['MAE ($)'].idxmin()
print(f"  ★ Best Model: {best} (lowest MAE)")


  Q4 SPENDING FORECAST — MODEL COMPARISON
              MAE ($)  RMSE ($)      R²
model                                  
Holt-Winters  3176.43   6291.57  0.6125
XGBoost         32.15    172.22  0.9906
LSTM           298.93   1058.49  0.7627

  ★ Best Model: XGBoost (lowest MAE)


In [28]:
# Save comparison table
comparison.to_csv('outputs/predictions/model_comparison.csv')

## Visualization: Forecast vs Actuals

In [29]:
# XGBoost predictions scatter
test_results = test_df.copy()
test_results['predicted'] = xgb_preds

# Aggregate to per-account Q4 totals
acct_actual = test_results.groupby('current_account_nbr')[target_col].sum()
acct_pred = test_results.groupby('current_account_nbr')['predicted'].sum()

scatter_df = pd.DataFrame({
    'actual': acct_actual,
    'predicted': acct_pred
}).reset_index()

# Merge segment info for coloring
scatter_df = scatter_df.merge(
    customer_base[['current_account_nbr']].drop_duplicates(),
    on='current_account_nbr', how='left'
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=scatter_df['actual'], y=scatter_df['predicted'],
    mode='markers', marker=dict(size=5, opacity=0.5, color='#3498db'),
    name='Predictions'
))

# Perfect prediction line
max_val = max(scatter_df['actual'].max(), scatter_df['predicted'].max())
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', line=dict(color='red', dash='dash', width=2),
    name='Perfect Prediction'
))

fig.update_layout(
    title='Q4 Forecast vs Actuals — XGBoost',
    xaxis_title='Actual Q4 Spend ($)',
    yaxis_title='Predicted Q4 Spend ($)',
    height=600, width=700,
    template='plotly_white'
)
fig.write_image("outputs/figures/q4_forecast_vs_actuals.png", scale=2)
fig.show()

## Save Q4 Predictions for Downstream Use

In [30]:
# Per-account Q4 forecast using best model (XGBoost)
q4_forecast = scatter_df[['current_account_nbr', 'predicted']].rename(
    columns={'predicted': 'q4_forecast'}
)
q4_forecast.to_parquet('outputs/predictions/q4_forecast.parquet', index=False)
print(f"Saved {len(q4_forecast)} account Q4 forecasts")

Saved 10771 account Q4 forecasts


In [ ]:
\